<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Anomaly Detection for Fraud and Sensor Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Rounak Biswas<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Load Libraries (2 Marks)

Import `numpy` as `np`, `pandas` as `pd`, `stats` from `scipy`, `IsolationForest` and `LocalOutlierFactor` from `sklearn.ensemble` and `sklearn.neighbors`, and `classification_report`, `precision_score`, `recall_score` from `sklearn.metrics`.

**Expected Output:** The code cell should execute without any errors.

In [60]:
import numpy as np
import pandas as pd

from scipy import stats

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score
)

### Question 2: Create the Dataset (4 Marks)

Generate a synthetic credit card transaction dataset. Set `np.random.seed(42)`. Create a DataFrame `normal` with 950 rows and a DataFrame `fraud` with 50 rows. Features should include `amount`, `hour_of_day`, `transactions_last_24h`, and `distance_from_home_km`, plus an `is_fraud` label (0 for normal, 1 for fraud). Combine them into a single DataFrame named `df`, shuffle using `.sample(frac=1, random_state=42)`, and reset the index.

**Hint:** Use `np.random.normal`, `np.random.randint`, `np.random.poisson`, and `np.random.exponential` to generate feature data.

**Expected Output:** Execution without errors, creating the `df` DataFrame.

In [61]:
np.random.seed(42)
# Normal Transactions (950 rows)
normal = pd.DataFrame({
    'amount': np.random.normal(loc=50, scale=15, size=950),
    'hour_of_day': np.random.randint(0, 24, size=950),
    'transactions_last_24h': np.random.poisson(lam=3, size=950),
    'distance_from_home_km': np.random.exponential(scale=5, size=950),
    'is_fraud': 0
})
# Fraud Transactions (50 rows)
fraud = pd.DataFrame({
    'amount': np.random.normal(loc=500, scale=150, size=50),
    'hour_of_day': np.random.randint(0, 24, size=50),
    'transactions_last_24h': np.random.poisson(lam=15, size=50),
    'distance_from_home_km': np.random.exponential(scale=50, size=50),
    'is_fraud': 1
})
# Combine and Shuffle
df = pd.concat([normal, fraud], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head(50)

,amount,hour_of_day,transactions_last_24h,distance_from_home_km,is_fraud
0,58.150403,17,1,3.400957,0
1,64.740365,10,2,6.331045,0
2,22.386887,18,0,0.275963,0
3,41.395070,0,0,2.323902,0
4,33.130369,1,3,3.217447,0
5,64.035176,14,1,4.127090,0
6,35.155928,18,5,11.666658,0
7,36.386545,6,3,0.952148,0
8,59.493977,4,0,0.784585,0
9,38.251201,18,0,5.705940,0


### Question 3: Check Dataset Shape and Distribution (2 Marks)

Print the shape of your combined DataFrame `df` and the value counts of the `is_fraud` column to observe the class distribution.

**Expected Output:** The shape (1000, 5) and the counts showing 950 normal (0) and 50 fraud (1) cases.

In [62]:
# Checking dataset shape
print("Dataset Shape:", df.shape)

# Checking class distribution
print("\nFraud Distribution:")
print(df['is_fraud'].value_counts())

Dataset Shape: (1000, 5)

Fraud Distribution:
is_fraud
0    950
1     50
Name: count, dtype: int64


### Question 4: Compare Feature Means by Class (3 Marks)

Group the dataset by the `is_fraud` column and calculate the mean values for all features (`amount`, `hour_of_day`, `transactions_last_24h`, `distance_from_home_km`). Print the resulting grouped means.

**Hint:** Use the `.groupby()` method and `.mean()`.

**Expected Output:** A table showing the mean feature values for normal (0) vs fraudulent (1) transactions.

In [63]:
# Comparing average feature values for normal vs fraud transactions
print(df.groupby('is_fraud').mean().to_string())

              amount  hour_of_day  transactions_last_24h  distance_from_home_km
is_fraud                                                                       
0          50.301261    11.394737               2.984211               5.035938
1         502.633444    12.080000              14.400000              51.798291


### Question 5: Apply Z-Score Anomaly Detection (3 Marks)

Compute the absolute Z-scores for the `distance_from_home_km` column. Create a new column named `zscore_anomaly` in `df` that contains `1` if the absolute Z-score is greater than 3, and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `np.abs(stats.zscore(...))`.

**Expected Output:** The count of anomalies flagged based on the distance feature.

In [64]:
# Computing absolute Z-scores for distance_from_home_km
z_scores = np.abs(stats.zscore(df['distance_from_home_km']))

# Flag anomalies (1 = anomaly, 0 = normal)
df['zscore_anomaly'] = (z_scores > 3).astype(int)

# Print total anomalies detected
print("Total anomalies flagged by Z-Score:",
      df['zscore_anomaly'].sum())

Total anomalies flagged by Z-Score: 19


### Question 6: Apply IQR Method (4 Marks)

Calculate the Interquartile Range (IQR) for the `amount` column. Identify bounds: `Lower = Q1 - 1.5 * IQR` and `Upper = Q3 + 1.5 * IQR`. Create a new column named `iqr_anomaly` in `df` containing `1` for values outside these bounds and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `.quantile(0.25)` for Q1 and `.quantile(0.75)` for Q3.

**Expected Output:** The total number of `amount` anomalies flagged by the IQR method.

In [65]:
# Here I am Calculating Q1, Q3 and IQR
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)

IQR = Q3 - Q1

# Defining lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag anomalies
df['iqr_anomaly'] = (
    (df['amount'] < lower_bound) |
    (df['amount'] > upper_bound)
).astype(int)

# Printing total anomalies detected
print("Total anomalies flagged by IQR:",
      df['iqr_anomaly'].sum())

Total anomalies flagged by IQR: 54


### Question 7: Train Isolation Forest (4 Marks)

Import `StandardScaler` from `sklearn.preprocessing` and scale the four feature columns. Then, create an `IsolationForest` model with `contamination=0.05` and `random_state=42`. Fit the model on the scaled features and add a column `isoforest_anomaly` to `df` containing `1` for anomalies and `0` for normal data.

**Hint:** `IsolationForest` returns `-1` for anomalies and `1` for normal points. Map these to `1` and `0` respectively.

**Expected Output:** The execution completes successfully.

In [66]:
from sklearn.preprocessing import StandardScaler

# Selecting feature columns
features = [
    'amount',
    'hour_of_day',
    'transactions_last_24h',
    'distance_from_home_km'
]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42
)

# Fit model and predict
iso_predictions = iso_forest.fit_predict(X_scaled)

# Convert predictions:
df['isoforest_anomaly'] = (iso_predictions == -1).astype(int)

print(df['isoforest_anomaly'].value_counts())


isoforest_anomaly
0    950
1     50
Name: count, dtype: int64


### Question 8: Evaluate Isolation Forest (3 Marks)

Use the `classification_report` function to evaluate the performance of your `isoforest_anomaly` predictions against the true `is_fraud` labels. Print the report.

**Expected Output:** A classification report displaying precision, recall, and f1-score for the model.

In [67]:
print(classification_report(
    df['is_fraud'],
    df['isoforest_anomaly']
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       950
           1       0.98      0.98      0.98        50

    accuracy                           1.00      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       1.00      1.00      1.00      1000



### Question 9: Local Outlier Factor Detection (3 Marks)

Train a `LocalOutlierFactor` model with `n_neighbors=20` and `contamination=0.05` on the scaled features. Add a new column `lof_anomaly` to `df` (where `1` indicates an anomaly and `0` indicates normal).

**Hint:** Use `.fit_predict()` to get the anomaly flags (similar to Isolation Forest, LOF returns `-1` for anomalies).

**Expected Output:** The execution completes successfully.

In [68]:
# Training Local Outlier Factor model
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

# Predicting anomalies
lof_predictions = lof.fit_predict(X_scaled)

# Converting predictions:
df['lof_anomaly'] = (lof_predictions == -1).astype(int)

print(df['lof_anomaly'].value_counts())

lof_anomaly
0    950
1     50
Name: count, dtype: int64


### Question 10: Evaluate LOF Model (2 Marks)

Calculate and print both the `precision_score` and `recall_score` for the `lof_anomaly` predictions against the true `is_fraud` labels.

**Expected Output:** Two numbers showing the precision and recall scores.

In [69]:
lof_precision = precision_score(
    df['is_fraud'],
    df['lof_anomaly']
)

lof_recall = recall_score(
    df['is_fraud'],
    df['lof_anomaly']
)

print("LOF Precision:", lof_precision)
print("LOF Recall:", lof_recall)

LOF Precision: 0.22
LOF Recall: 0.22
